This final notebook is just to tie up some loose ends. We need to ensure the FWER for both cohorts ran the same amount of times, that the boostrap nullMethod doesn't significantly affect our results, and the genome-wide concordance scan to see if we missed something.

In [4]:
setwd('/home/ethan-xiao/food-allergy-biomarkers/data')
getwd()
library(GenomicRanges)

[1] "/home/ethan-xiao/food-allergy-biomarkers/data"

Loading required package: stats4

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min


Loading required package: S4Vectors


Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The follo

Checks every bumphunter region in the infant cohort against every region in the adolescent cohort, rather than only checking ISG15 and nc886 by hand. Answers how many other loci show the same cross-cohort, direction-matched pattern ISG15 does and where ISG15 ranks among them.

In [6]:
bumps_infant <- readRDS("bumphunter_114134.rds")$table
bumps_adol   <- readRDS("bumphunter_189148_B250.rds")$table

dim(bumps_infant)
dim(bumps_adol)

[1] 38638    14

[1] 80793    14

In [7]:
gr_infant <- GRanges(seqnames = bumps_infant$chr,
                      ranges = IRanges(start = bumps_infant$start, end = bumps_infant$end),
                      value = bumps_infant$value,
                      p.value = bumps_infant$p.value,
                      fwer = bumps_infant$fwer)

gr_adol <- GRanges(seqnames = bumps_adol$chr,
                    ranges = IRanges(start = bumps_adol$start, end = bumps_adol$end),
                    value = bumps_adol$value,
                    p.value = bumps_adol$p.value,
                    fwer = bumps_adol$fwer)

gr_infant

GRanges object with 38638 ranges and 3 metadata columns:
          seqnames              ranges strand |     value     p.value      fwer
             <Rle>           <IRanges>  <Rle> | <numeric>   <numeric> <numeric>
      [1]     chr5 135414858-135416613      * |  0.410586 2.97993e-06      0.08
      [2]    chr12       739953-740338      * |  0.685943 1.63896e-05      0.34
      [3]     chr2   30669597-30669863      * |  0.769272 2.01145e-05      0.40
      [4]     chr8 144660395-144660772      * | -0.515112 3.79941e-05      0.70
      [5]    chr15   22833172-22833400      * | -0.449823 7.07734e-05      0.88
      ...      ...                 ...    ... .       ...         ...       ...
  [38634]     chr8           121714454      * | -0.150009    0.999757         1
  [38635]    chr16            86589283      * |  0.150005    0.999870         1
  [38636]    chr10             7215201      * |  0.150005    0.999880         1
  [38637]    chr21            34441330      * |  0.150002    0.

In [8]:
hits <- findOverlaps(gr_infant, gr_adol)

length(hits) #Finding concordance 

[1] 7703

In [9]:
overlap_table <- data.frame(
  chr           = as.character(seqnames(gr_infant))[queryHits(hits)],
  start_infant  = start(gr_infant)[queryHits(hits)],
  end_infant    = end(gr_infant)[queryHits(hits)],
  value_infant  = gr_infant$value[queryHits(hits)],
  p_infant      = gr_infant$p.value[queryHits(hits)],
  start_adol    = start(gr_adol)[subjectHits(hits)],
  end_adol      = end(gr_adol)[subjectHits(hits)],
  value_adol    = gr_adol$value[subjectHits(hits)],
  p_adol        = gr_adol$p.value[subjectHits(hits)]
)

nrow(overlap_table)
head(overlap_table)

[1] 7703

,chr,start_infant,end_infant,value_infant,p_infant,start_adol,end_adol,value_adol,p_adol
,<chr>,<int>,<int>,<dbl>,<dbl>,<int>,<int>,<dbl>,<dbl>
1,chr5,135414858,135416613,0.4105861,2.979933e-06,135415190,135416613,-0.9733978,9.399045e-07
2,chr12,739953,740338,0.6859428,1.638963e-05,739953,740338,0.4937805,2.455783e-04
3,chr2,30669597,30669863,0.7692717,2.011455e-05,30669597,30669863,0.5455590,2.833248e-04
4,chr8,144660395,144660772,-0.5151121,3.799415e-05,144660590,144660631,0.3148713,1.852702e-03
5,chr8,144660395,144660772,-0.5151121,3.799415e-05,144660772,144660772,0.2292614,3.398028e-01
6,chr15,22833172,22833400,-0.4498230,7.077341e-05,22833172,22833213,-0.2002484,5.823329e-02


In [10]:
overlap_table$direction_matched <- sign(overlap_table$value_infant) == sign(overlap_table$value_adol)

concordant <- overlap_table[overlap_table$direction_matched, ]
concordant$combined_p <- pchisq(
  -2 * (log(concordant$p_infant) + log(concordant$p_adol)),
  df = 4, lower.tail = FALSE
)
concordant <- concordant[order(concordant$combined_p), ]

nrow(concordant)
head(concordant, 10)

[1] 4410

,chr,start_infant,end_infant,value_infant,p_infant,start_adol,end_adol,value_adol,p_adol,direction_matched,combined_p
,<chr>,<int>,<int>,<dbl>,<dbl>,<int>,<int>,<dbl>,<dbl>,<lgl>,<dbl>
2,chr12,739953,740338,0.6859428,1.638963e-05,739953,740338,0.4937805,2.455783e-04,TRUE,8.183002e-08
3,chr2,30669597,30669863,0.7692717,2.011455e-05,30669597,30669863,0.5455590,2.833248e-04,TRUE,1.138820e-07
11,chr8,11666017,11666810,-0.3315998,1.564465e-04,11666281,11666594,-0.4614894,1.806121e-04,TRUE,5.194028e-07
30,chr17,180404,181288,0.4456291,3.091681e-04,180404,181288,0.6855639,1.177136e-04,TRUE,6.597698e-07
397,chr17,33759512,33759573,-0.3102975,4.846861e-03,33759512,33760971,-0.5305645,8.947891e-06,TRUE,7.786294e-07
288,chr17,33759957,33759986,-0.2364414,5.524051e-03,33759512,33760971,-0.5305645,8.947891e-06,TRUE,8.809530e-07
27,chr15,81426347,81426669,-0.2352937,3.263027e-04,81426347,81426669,-0.3349684,1.773788e-04,TRUE,1.022430e-06
19,chr12,122356390,122356598,-0.5052124,1.817759e-04,122356390,122356781,-0.4530535,3.303576e-04,TRUE,1.058584e-06
87,chr4,99064102,99064573,-0.2428453,1.478792e-03,99064102,99064904,-0.5219703,4.316042e-05,TRUE,1.121226e-06


In [11]:
concordant$rank <- seq_len(nrow(concordant))

isg15_hit <- concordant[concordant$chr == "chr1" &
                         concordant$start_infant >= 948000 &
                         concordant$end_infant <= 950000, ]
print(isg15_hit)

     chr start_infant end_infant value_infant   p_infant start_adol end_adol
781 chr1       948625     948627   -0.2232425 0.01749817     948625   948627
    value_adol      p_adol direction_matched  combined_p rank
781 -0.3722121 0.005893089              TRUE 0.001049706  218


Pulls the gene(s) overlapping each of the top 20 concordant regions by combined p-value, using the EPIC array annotation. This is the manual, per-row version; the full genome-wide version (all 4,392 regions) is done further below.

In [12]:
library(IlluminaHumanMethylationEPICanno.ilm10b4.hg19)
ann <- getAnnotation(IlluminaHumanMethylationEPICanno.ilm10b4.hg19)

top20 <- head(concordant, 20)
top20$genes <- sapply(seq_len(nrow(top20)), function(i) {
  region_probes <- rownames(ann)[ann$chr == top20$chr[i] &
                                  ann$pos >= top20$start_infant[i] &
                                  ann$pos <= top20$end_infant[i]]
  paste(unique(ann[region_probes, "UCSC_RefGene_Name"]), collapse = "; ")
})
print(top20[, c("chr", "start_infant", "end_infant", "combined_p", "genes")])

Loading required package: minfi

Loading required package: SummarizedExperiment

Loading required package: MatrixGenerics

Loading required package: matrixStats


Attaching package: ‘MatrixGenerics’


The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQRDiffs, rowIQRs, rowLogSumExps,
    rowMadDiffs, rowMads, rowMaxs, rowMeans2, rowMedians, rowMins,
    rowOrderStats, 

      chr start_infant end_infant   combined_p
2   chr12       739953     740338 8.183002e-08
3    chr2     30669597   30669863 1.138820e-07
11   chr8     11666017   11666810 5.194028e-07
30  chr17       180404     181288 6.597698e-07
397 chr17     33759512   33759573 7.786294e-07
288 chr17     33759957   33759986 8.809530e-07
27  chr15     81426347   81426669 1.022430e-06
19  chr12    122356390  122356598 1.058584e-06
87   chr4     99064102   99064573 1.121226e-06
303  chr7       872130     872208 1.239421e-06
71   chr1    248100345  248100614 1.559740e-06
985 chr11     45407537   45407537 1.565668e-06
55   chr4       124232     124344 1.627315e-06
200 chr11     74988026   74988026 1.745318e-06
65   chr1      2084319    2084595 2.820209e-06
91  chr10    135341870  135342560 3.987975e-06
131 chr19     13875014   13875289 4.213830e-06
92  chr10    135341870  135342560 4.559008e-06
44   chr5    176797920  176798049 4.564031e-06
239  chr4     69435250   69435601 4.709099e-06
             

The raw overlap scan can count the same physical region more than once (e.g., one infant-cohort region overlapping two adjacent adolescent-cohort sub-regions). This collapses each unique locus down to its single best-matching overlap before ranking (with some mishaps along the way)

In [13]:
#Checking multiple overlaps

library(dplyr)

concordant_deduped <- concordant %>%
  group_by(chr, start_infant, end_infant) %>%
  slice_min(combined_p, n = 1) %>%
  ungroup() %>%
  arrange(combined_p)

concordant_deduped$rank <- seq_len(nrow(concordant_deduped))
nrow(concordant_deduped)

isg15_dedup <- concordant_deduped[concordant_deduped$chr == "chr1" &
                                   concordant_deduped$start_infant >= 948000 &
                                   concordant_deduped$end_infant <= 950000, ]
print(isg15_dedup)


Attaching package: ‘dplyr’


The following object is masked from ‘package:minfi’:

    combine


The following objects are masked from ‘package:Biostrings’:

    collapse, intersect, setdiff, setequal, union


The following object is masked from ‘package:XVector’:

    slice


The following object is masked from ‘package:Biobase’:

    combine


The following object is masked from ‘package:matrixStats’:

    count


The following objects are masked from ‘package:GenomicRanges’:

    intersect, setdiff, union


The following object is masked from ‘package:Seqinfo’:

    intersect


The following objects are masked from ‘package:IRanges’:

    collapse, desc, intersect, setdiff, slice, union


The following objects are masked from ‘package:S4Vectors’:

    first, intersect, rename, setdiff, setequal, union


The following objects are masked from ‘package:BiocGenerics’:

    combine, intersect, setdiff, setequal, union


The following object is masked from ‘package:generics’:

    explai

[1] 4392

# A tibble: 1 × 12
  chr   start_infant end_infant value_infant p_infant start_adol end_adol
  <chr>        <int>      <int>        <dbl>    <dbl>      <int>    <int>
1 chr1        948625     948627       -0.223   0.0175     948625   948627
# ℹ 5 more variables: value_adol <dbl>, p_adol <dbl>, direction_matched <lgl>,
#   combined_p <dbl>, rank <int>


In [14]:
isg15_dedup$rank

[1] 215

In [15]:
write.csv(top20, "top20_candidates_for_rnaseq_check.csv", row.names = FALSE)

Same idea as the top-20 annotation above, but using GRanges/findOverlaps to annotate all 4,392 candidates at once instead of looping row-by-row. Output is written to all_candidates_for_rnaseq_check.csv for the RNA-seq cross-validation done in notebook 10.

In [16]:
library(IlluminaHumanMethylationEPICanno.ilm10b4.hg19)
ann <- getAnnotation(IlluminaHumanMethylationEPICanno.ilm10b4.hg19)

#Build one GRanges object for all 4,392 candidate regions at once
gr_candidates <- GRanges(seqnames = concordant_deduped$chr,
                          ranges = IRanges(start = concordant_deduped$start_infant,
                                            end = concordant_deduped$end_infant))

#Build one GRanges object for every probe on the array (each probe is a single position)
gr_probes <- GRanges(seqnames = ann$chr,
                      ranges = IRanges(start = ann$pos, width = 1),
                      gene = ann$UCSC_RefGene_Name)

#One overlap search finds every probe inside every candidate region, all at once
probe_hits <- findOverlaps(gr_candidates, gr_probes)

#Group the matched genes by which candidate region they belong to
gene_by_candidate <- split(gr_probes$gene[subjectHits(probe_hits)], queryHits(probe_hits))

concordant_deduped$genes <- sapply(seq_len(nrow(concordant_deduped)), function(i) {
  g <- gene_by_candidate[[as.character(i)]]
  if (is.null(g)) return("")
  paste(unique(g), collapse = "; ")
})

write.csv(concordant_deduped, "all_candidates_for_rnaseq_check.csv", row.names = FALSE)

In [17]:
dim(concordant_deduped)
head(concordant_deduped[, c("chr", "start_infant", "end_infant", "genes")])

[1] 4392   13

chr,start_infant,end_infant,genes
<chr>,<int>,<int>,<chr>
chr12,739953,740338,NINJ2; LOC100049716;NINJ2;NINJ2
chr2,30669597,30669863,LCLAT1;LCLAT1; LCLAT1;LCLAT1;LCLAT1;LCLAT1
chr8,11666017,11666810,FDFT1; FDFT1;FDFT1;FDFT1;FDFT1;FDFT1;FDFT1;FDFT1;FDFT1;FDFT1;FDFT1;FDFT1; FDFT1;FDFT1;FDFT1;FDFT1;FDFT1;FDFT1;FDFT1;FDFT1;FDFT1;FDFT1;FDFT1;FDFT1
chr17,180404,181288,RPH3AL; RPH3AL;RPH3AL;RPH3AL;RPH3AL;LOC100506388;LOC100506388; LOC100506388;LOC100506388;LOC100506388;RPH3AL;RPH3AL;RPH3AL;RPH3AL
chr17,33759512,33759573,SLFN12; SLFN12;SLFN12
chr17,33759957,33759986,SLFN12


Checks bumps_adol directly for any region clearing FWER < 0.05.

In [18]:
#did the adolescent cohort (on its own) have any bump reach FWER < 0.05?
min(bumps_adol$fwer)
sum(bumps_adol$fwer < 0.05)
bumps_adol[bumps_adol$fwer < 0.05, ]

[1] 0.1

[1] 0

chr,start,end,value,area,cluster,indexStart,indexEnd,L,clusterL,p.value,fwer,p.valueArea,fwerArea
<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<int>,<int>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>


Check promising candidates from work done in Notebook 10

The infant cohort's original analysis used B=50 permutations. It was rerun at B=250 by editing the original cell in notebook 6 in place, which overwrote both the saved `.rds` file and that cell's recorded output, so the B=50 result is no longer independently reproducible from any file in this repo. The original values (ISG15 p=0.0175) are preserved in the project's abstract draft, dated prior to Sept 1, 2026. B=250 is the adopted, final result used throughout this project. This section exists to document that the update was checked and made no meaningful difference.

In [3]:
bumps_infant_b250 <- readRDS(file.path('/home/ethan-xiao/food-allergy-biomarkers/data/bumphunter_114134.rds'))

isg15_b250 <- subset(bumps_infant_b250$table, chr == 'chr1' & start < 950000 & end > 948000)
rgs14_b250 <- subset(bumps_infant_b250$table, chr == 'chr5' & start < 176799000 & end > 176797000)

print(isg15_b250)
print(rgs14_b250)

       chr  start    end      value      area cluster indexStart indexEnd L
16453 chr1 948625 948627 -0.2232425 0.4464850      93        281      282 2
16454 chr1 948814 948814 -0.2133394 0.2133394      93        284      284 1
      clusterL   p.value fwer p.valueArea fwerArea
16453       12 0.0176013    1  0.02787834        1
16454       12 0.2269600    1  0.25345087        1
       chr     start       end   value    area cluster indexStart indexEnd L
12923 chr5 176797920 176798049 0.52362 1.57086  306029     602877   602879 3
      clusterL      p.value fwer p.valueArea fwerArea
12923        5 0.0002627308    1 0.001524383        1


In [6]:
concordant_deduped <- read.csv(file.path('/home/ethan-xiao/food-allergy-biomarkers/data/all_candidates_for_rnaseq_check.csv'))
dim(concordant_deduped)

[1] 4392   13

Checks whether the 8 genes that passed genome-wide FDR correction in the RNA-seq analysis (independent of methylation finding) show up among the methylation-concordant candidates and whether the array ehas probe coverage near them.

In [7]:
genes_of_interest <- c("PTGS2", "VEGFC", "SLC22A5", "ARHGEF37", "LSM2", "C9orf72", "RAI1", "CARD8-AS1", "CARD8")

matches <- concordant_deduped[sapply(strsplit(concordant_deduped$genes, "; "),
                                       function(g) any(trimws(g) %in% genes_of_interest)), ]
print(matches)

       chr start_infant end_infant value_infant   p_infant start_adol end_adol
1096 chr17     17626019   17626019   -0.3448798 0.02039764   17626019 17626019
     value_adol    p_adol direction_matched combined_p rank genes
1096 -0.2233683 0.3686523              TRUE 0.04429245 1096  RAI1


In [9]:
library(IlluminaHumanMethylationEPICanno.ilm10b4.hg19)
ann <- getAnnotation(IlluminaHumanMethylationEPICanno.ilm10b4.hg19)

Loading required package: minfi

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min


Loading required package: GenomicRanges

Loading required package: stats4

Loading required package: S4Vectors


Attaching package: ‘S4Vectors’


The fol

In [10]:
ann_genes_split <- strsplit(ann$UCSC_RefGene_Name, ";")
has_gene <- sapply(ann_genes_split, function(g) any(trimws(g) %in% genes_of_interest))

coverage_check <- ann[has_gene, c("Name", "chr", "pos", "UCSC_RefGene_Name")]
table(coverage_check$UCSC_RefGene_Name)


                                                                                       ARHGEF37 
                                                                                             19 
                                                                              ARHGEF37;ARHGEF37 
                                                                                              1 
                                                                                        C9orf72 
                                                                                              1 
                                                                                C9orf72;C9orf72 
                                                                                              6 
                                                                        C9orf72;C9orf72;C9orf72 
                                                                                              7 
                             

- 4,392 direction-matched, cross-cohort concordant methylation regions identified genome-wide (deduplicated by locus).
- ISG15 ranks 215th of 4,392 by combined methylation evidence
- RGS14 ranks 18th of 4,392, so it's the strongest methylation-only candidate found in this scan, though not previously connected to food allergy. Cross-omics validation against RNA-seq (carried out in notebook 10) finds that RGS14 ranks 1st of 1,584 on a combined methylation+RNA-seq metric, versus ISG15's 49th.
- The adolescent cohort alone has zero regions clearing genome-wide FWER < 0.05, consistent with the infant cohort's best result (nc886, later excluded as a known artifact) also failing correction.
- Of the 8 genes independently significant in the RNA-seq analysis, only RAI1 shows any methylation-side concordance; PTGS2 (the most allergy-relevant of the 8) has array coverage but no concordant methylation signal, consistent with it being an activation-responsive rather than resting-state gene.
- The infant cohort's B=50→B=250 permutation rematch showed no meaningful change in either ISG15 or RGS14's results (see note above on file provenance).

Continued in notebook 10 (RNA-seq cross-validation) and notebook 6 (bootstrap nullMethod check).